# Release stale Cluster connection

Utility notebook for the `ValueError: invalid literal for int() with base 10: ''` failure inside `HardwareAgent.connect_clusters()` (raised from `qblox_instruments`'s SCPI transport). This happens when a socket to the Cluster is left in a bad state — usually because another process (or an interrupted cell in this same kernel) still holds a connection, but it can also be a module-level fault. Run the steps below top to bottom.

In [1]:
import json
import os
import signal
import subprocess
from pathlib import Path

from qblox_lab.config.hardware import create_hardware_agent
from qblox_lab.config.sessions import SESSIONS
from qcodes.instrument import Instrument

In [2]:
SESSION = SESSIONS["AS_QRC"]  # change if you use a different session
hw_cfg = json.loads(Path(SESSION.hardware_config).read_text())
cluster_ips = sorted({v["ip"] for v in hw_cfg.values() if isinstance(v, dict) and "ip" in v})
print("Cluster IPs:", cluster_ips)

Cluster IPs: []


## Step 1 — diagnose *before* touching any connections

The traceback lands in `_create_cluster` → `Cluster(...)` construction itself, on a **per-module** query (`AFE:ATT:OUT0:MAX?`, sent via `_slot_connection(slot)` while probing attenuation ranges for the RF modules). That's different from a generic "socket already in use" failure — it points at one specific module, so check per-slot fault flags first using a raw `qblox_instruments.native.Cluster` connection (bypasses `qblox_scheduler`/QCoDeS entirely, so it can't collide with anything step 2 does).

In [3]:
from qblox_instruments.native import Cluster as NativeCluster

raw = NativeCluster(cluster_ips[0])
try:
    print("System status:", raw.get_system_status())
    try:
        state = raw._get_cluster_state()
        print("\nCluster-level flags:", state.flags)
        for slot, mod_state in sorted(state.modules.items()):
            print(f"  slot {slot}: {mod_state}")
    except Exception as exc:
        print("_get_cluster_state() unavailable (needs firmware >= 2.1.0):", exc)
finally:
    raw.close()

IndexError: list index out of range

Check the per-slot flags above for the slots used in `hw_cfg` (e.g. slot 6 / slot 8 for the `AS_QRC` session's `QCM_RF`/`QRM_RF`). A fault/error flag on one of them (not fully booted, AFE not ready, thermal, ...) explains the empty SCPI response directly — the fix there is rebooting *that module* from the Cluster's web UI, not chasing another process. If both come back clean, move on to steps 2–4 below (connection contention).

## Step 2 — release any stale connection held by *this* kernel

If a previous cell in this same notebook created a `Cluster`/`HardwareAgent` and got interrupted (e.g. `KeyboardInterrupt`) before it could close cleanly, its socket can still be open here. This is safe to run any time, even if nothing is stale.

In [4]:
Instrument.close_all()
print("Closed all QCoDeS instruments cached in this kernel.")

Closed all QCoDeS instruments cached in this kernel.


## Step 3 — find *other* processes still connected to the Cluster

Lists TCP connections to the Cluster's IP(s) together with the owning PID/process name (needs `ss` to be able to see other users' sockets, otherwise the PID column shows blank — rerun with `sudo` if so).

In [5]:
ss_output = subprocess.run(["ss", "-tnp"], capture_output=True, text=True).stdout
for ip in cluster_ips:
    print(f"\n--- connections to {ip} ---")
    matches = [line for line in ss_output.splitlines() if ip in line]
    print("\n".join(matches) if matches else "(none found)")

If step 3 shows a PID that belongs to another *live* script/kernel, prefer going back to that process and running `Instrument.close_all()` there (same as step 2) rather than killing it.

Only use the cell below if that process is unresponsive (e.g. a crashed/zombie kernel) — it sends `SIGTERM` to just that PID, not to whatever else may be running. Fill in the PID from step 3's output first; it is intentionally left unexecuted.

In [ ]:
# PID = 12345  # <- set this from step 3's output, then uncomment the line below
# os.kill(PID, signal.SIGTERM)

## Step 4 — verify the Cluster accepts a fresh connection

In [6]:
hardware_agent = create_hardware_agent(
    hardware_configuration=SESSION.hardware_config,
    device_configuration=SESSION.device_config,
    output_dir=Path("data"),
    create_dummy_connections=False,
)
hardware_agent.connect_clusters()
print("Reconnected successfully.")

ValueError: invalid literal for int() with base 10: ''